# Resumen de 10 artículos de tecnología (115661)
Lee `noticias.csv`, descarga cada página con `requests` + `BeautifulSoup`, extrae los párrafos `<p>`, los resume con `summarization` y guarda `resumenes.csv`.

## 1. Función summarization

In [ ]:
import string
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize

for recurso in ("punkt", "punkt_tab", "stopwords", "wordnet"):
    nltk.download(recurso, quiet=True)


def summarization(text, n_sentences=3, language="english"):
    """Devuelve un resumen con las n frases mas relevantes, en su orden original."""
    sentences = sent_tokenize(text, language=language)
    if len(sentences) <= n_sentences:
        return text

    stop_words = set(stopwords.words(language))
    # WordNet solo lematiza ingles; en otros idiomas se usa la palabra tal cual.
    lemmatizer = WordNetLemmatizer() if language == "english" else None

    def clean_words(sentence):
        words = word_tokenize(sentence.lower(), language=language)
        words = [w for w in words if w.isalpha() and w not in stop_words]
        return [lemmatizer.lemmatize(w) for w in words] if lemmatizer else words

    words_per_sentence = [clean_words(s) for s in sentences]
    frequencies = Counter(w for words in words_per_sentence for w in words)

    # Puntuacion: suma de frecuencias normalizada por longitud (no favorece frases largas).
    scores = {}
    for i, words in enumerate(words_per_sentence):
        if words:
            scores[i] = sum(frequencies[w] for w in words) / len(words)

    best = sorted(scores.items(), key=lambda item: item[1], reverse=True)[:n_sentences]
    selected_sentences = sorted(best)  # orden original del texto
    return " ".join([sentences[i] for i, _ in selected_sentences])

## 2. Archivo de enlaces
Si no tienes `noticias.csv`, esta celda lo crea.

In [ ]:
%%writefile noticias.csv
titulo,url
Inteligencia artificial,https://en.wikipedia.org/wiki/Artificial_intelligence
Aprendizaje automatico,https://en.wikipedia.org/wiki/Machine_learning
Computacion cuantica,https://en.wikipedia.org/wiki/Quantum_computing
Blockchain,https://en.wikipedia.org/wiki/Blockchain
Internet de las cosas,https://en.wikipedia.org/wiki/Internet_of_things
Computacion en la nube,https://en.wikipedia.org/wiki/Cloud_computing
Redes 5G,https://en.wikipedia.org/wiki/5G
Vehiculo autonomo,https://en.wikipedia.org/wiki/Self-driving_car
Realidad virtual,https://en.wikipedia.org/wiki/Virtual_reality
Ciberseguridad,https://en.wikipedia.org/wiki/Computer_security

## 3. Descargar, resumir y guardar

In [ ]:
import csv
import re

import requests
from bs4 import BeautifulSoup


# Wikipedia pide un User-Agent identificable
HEADERS = {"User-Agent": "Mozilla/5.0 (tarea-escolar-kodland)"}


def extraer_texto(url):
    """Descarga la pagina y une el texto de todos los parrafos <p>."""
    respuesta = requests.get(url, headers=HEADERS, timeout=30)
    respuesta.raise_for_status()
    soup = BeautifulSoup(respuesta.text, "html.parser")
    texto = " ".join(p.get_text(" ", strip=True) for p in soup.find_all("p"))
    return re.sub(r"\[ ?\w+ ?\]", "", texto)  # quita citas tipo [ 12 ]



def main(entrada="noticias.csv", salida="resumenes.csv", n_frases=5):
    with open(entrada, newline="", encoding="utf-8") as f:
        filas = list(csv.DictReader(f))

    resultados = []
    for fila in filas:
        try:
            resumen = summarization(extraer_texto(fila["url"]), n_frases)
        except requests.RequestException as error:
            resumen = f"ERROR al descargar: {error}"
        print(f"{fila['titulo']}: {len(resumen)} caracteres")
        resultados.append({"titulo": fila["titulo"], "url": fila["url"], "resumen": resumen})

    with open(salida, "w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=["titulo", "url", "resumen"])
        escritor.writeheader()
        escritor.writerows(resultados)


main()

## 4. Ver el resultado

In [ ]:
import pandas as pd

pd.read_csv('resumenes.csv')